**S24 Pipeline v2**

Changes:
1) setup_paths def at S24/utils to automatically create paths and handle an incoming sysml file 
2)

In [ ]:
import os, sys
sys.path.insert(0, "/home/ang/usd_ang/lib/python")
os.environ["LD_LIBRARY_PATH"] = "/home/ang/usd_ang/lib"

**Load or search for a sysml file and create paths**

In [ ]:
from S24.utils import setup_paths

my_sysml = "LunarSpaceport1.sysml"
paths = setup_paths(sysml_file=my_sysml,use_repo=True)
for key, value in paths.items():
    print(f"{key}: {value}")

**Export sysml->json/jsons->python object**

In [ ]:
from __future__ import annotations

import json
from typing import Any, Dict, List, Optional

from S24.sysml.ast import Model, PartNode
from S24.sysml.evaluator import evaluate_attributes, build_part_json, parse_sysml

In [ ]:

def sysml_to_json(sysml_text: str, *, namespace: str = "lunarspaceport1") -> List[Dict[str, Any]]:
    """
    High-level API:
      SysML text -> parsed model -> evaluated -> flat list JSON with parent/children.
    """
    model: Model = parse_sysml(sysml_text)
    evaluate_attributes(model)

    results: List[Dict[str, Any]] = []

    def emit_part(part: PartNode, parent: Optional[PartNode] = None) -> None:
        obj = build_part_json(part, namespace=namespace)

        if parent is not None:
            obj["parent"] = parent.name

        child_names = [n for n in part.children.keys() if not n.lower().endswith("dims")]
        if child_names:
            obj["children"] = child_names

        results.append(obj)

        for child_name, child in part.children.items():
            if child_name.lower().endswith("dims"):
                continue
            emit_part(child, parent=part)

    for top in model.parts.values():
        emit_part(top, parent=None)

    return results


def write_json(parts: List[Dict[str, Any]], output_path: str) -> str:
    """
    Write a parts list to disk as pretty JSON.
    Returns output_path.
    """
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(parts, f, indent=2)
    return output_path

In [ ]:
# from S24.sysml import sysml_to_json, write_json

SYSML_FILE = paths["SYSML_FILE"]
JSON_FILE = paths["JSON_FILE"]

with open(SYSML_FILE, "r", encoding="utf-8") as f:
    sysml_text = f.read()

# print(sysml_text[:], "...")

parts_json = sysml_to_json(
    sysml_text,
    namespace="lunarspaceport1"
)

# write_json(parts_json, JSON_FILE)

# len(parts_json)
# parts_json[0]

**Turn JSON into an structured python object and vet json graph validity and entries for usd**

In [ ]:
# from S24.jsonio import VettingProc

# vetting = VettingProc(source=str(JSON_FILE))
# vetted_parts = vetting.by_name

# list(vetted_parts.keys())

In [ ]:
# from S24.usd import USDBuilder

# builder = USDBuilder(
#     vetted_parts,
#     overwrite=True,
#     use_paths_from_vetted=False
# )

# outputs = builder.build_all_parts()
# outputs

# scene_path = builder.write_assembly_scene(
#     root_name="HabitationModule",
#     include_root_as_instance=True,
#     instanceable=False,
#     debug_refs=True
# )

# scene_path